# This method attemps to gain a higher accuracy by training a ResNet model on Amazon-Berkley Objects dataset since the ABO dataset contains object images and their respective weight

# Download Dataset

In [42]:
import os

# 1. Download and extract listings metadata (approx 83 MB)
if not os.path.exists('listings'):
    print("Downloading listings metadata...")
    !wget https://amazon-berkeley-objects.s3.amazonaws.com/archives/abo-listings.tar
    !tar -xf abo-listings.tar
else:
    print("Listings metadata already exists, skipping download...")

# 2. Download and extract downscaled images (approx 3 GB)
if not os.path.exists('images'):
    print("Downloading downscaled images...")
    !wget https://amazon-berkeley-objects.s3.amazonaws.com/archives/abo-images-small.tar
    !tar -xf abo-images-small.tar
else:
    print("Images already exist, skipping download...")

Listings metadata already exists, skipping download...
Images already exist, skipping download...


In [43]:
import torch
import torchvision.models as models
from torch import nn
import torch.nn.functional as F
from typing import Dict, Union, List

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [47]:
import pandas as pd

# Load data sets
metadata_df = pd.read_json('listings/metadata/listings_0.json.gz', lines=True, compression='gzip')
images_df = pd.read_csv('images/metadata/images.csv.gz', compression='gzip')

# Filter rows to get rid of NaN values in 'item_weight' and 'item_name' columns
metadata_df = metadata_df.dropna(subset=['item_weight', 'item_name']).copy()

# Create new columns directly in the existing dataframe to make sure rows stay aligned
metadata_df['weight'] = metadata_df['item_weight'].str[0].str['value']
metadata_df['unit'] = metadata_df['item_weight'].str[0].str['unit']
metadata_df['label'] = metadata_df['item_name'].str[0].str['value']

# Basically tape both dataframes together where the images id's match
df = pd.merge(
    metadata_df[['item_id', 'weight', 'unit', 'label', 'main_image_id']], 
    images_df, 
    left_on='main_image_id', 
    right_on='image_id', 
    how='left'
)

# Final image path cleanup
df['path'] = 'images/small/' + df['path']

# Keep only columns we need
df = df[['item_id', 'weight', 'unit', 'label', 'main_image_id', 'path', 'height', 'width']]

print(df.head())

      item_id  weight    unit  \
0  B07P8ML82R    1.45  pounds   
1  B07H9GMYXS    2.20  pounds   
2  B07CTPR73M    0.10  pounds   
3  B01MTEI8M6    6.70  ounces   
4  B0853X2F4M   50.00   grams   

                                               label main_image_id  \
0  22" Bottom Mount Drawer Slides, White Powder C...   619y9YG9cnL   
1  AmazonBasics PETG 3D Printer Filament, 1.75mm,...   81NP7qh2L6L   
2       Stone & Beam Stone Brown Swatch, 25020039-01   61Rp4qOih9L   
3  The Fix Amazon Brand Women's French Floral Emb...   714CmIfKIYL   
4  Amazon Brand - Solimo Designer Autumn Girl 3D ...   81+4dBN1jsL   

                           path  height   width  
0  images/small/9f/9f76d27b.jpg  1200.0  1200.0  
1  images/small/66/665cc994.jpg  2492.0  2492.0  
2  images/small/b4/b4f9d0cc.jpg   500.0   500.0  
3  images/small/2b/2b1c2516.jpg   868.0  1779.0  
4  images/small/9d/9dfccb37.jpg  2200.0  1879.0  
